# C1: XLM-RoBERTa on Sinhala-English code-mixed citizen messages (v2)

**v2 changes** (v1 = the notebook behind Observations 5–7, kept unchanged):
- Intent reads the **last 3 user messages** (window_3, the setting chosen in Observation 8), not the full history
- Long inputs are cut from the **left**, so the newest message is never dropped
- Trains on all intents in the silver set (all 12, once the support-intent dialogues are added)
- Optional **Weights & Biases** logging (`USE_WANDB`)
- Saves to `MyDrive/c1_models/run_v2_seed<SEED>/` and `xlmr_results_v2_seed<SEED>.json`, so v1 files are never overwritten

Trains three models and tests them on the **hand-checked gold set**:

| Task | What it predicts | Train on | Test on |
|---|---|---|---|
| A. Language ID | SI / EN / MIXED / NE / OTHER for every word | `silver_tokens.jsonl` (synthetic) | `gold_tokens.jsonl` |
| B. Entities | district, date, travel_date, new_value, ... (BIO tags) | same | same |
| C. Intent | which service the citizen wants, from the last 3 user messages | same | gold turns whose intent is in training |

**Rule:** gold data is never used for training or model selection. A dev split of the
synthetic dialogues picks the best epoch.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → Run all.
Upload these 4 files when asked (from `research/`):
`data/synthetic/silver_tokens.jsonl`, `data/gold_tokens.jsonl`, `data/fresh_tokens.jsonl`, `results/baseline_fasttext.json`
Allow Google Drive access when asked (the models are saved there). If W&B asks for an API key,
paste it from wandb.ai/authorize (free account); set `C1_WANDB=0` to skip W&B.

In [ ]:
# Setup
import json, os, random, time, collections
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModelForSequenceClassification, \
    get_linear_schedule_with_warmup

MODEL = os.environ.get("C1_MODEL", "xlm-roberta-base")
EPOCHS = int(os.environ.get("C1_EPOCHS", 8))
BATCH, MAX_LEN = 16, 128
SEED = int(os.environ.get("C1_SEED", 42))      # change to 1, 2, ... for repeat runs
SAVE_MODELS = os.environ.get("C1_SAVE", "1") == "1"   # saves the 3 models to Google Drive at the end
INTENT_WINDOW = int(os.environ.get("C1_WINDOW", 3))   # user messages the intent model reads (0 = all)
USE_WANDB = os.environ.get("C1_WANDB", "1") == "1"    # log to Weights & Biases (asks for your API key once)
RUN_TAG = "v2"
LR = float(os.environ.get("C1_LR", 3e-5))
DEV_SHARE = 0.1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| model:", MODEL, "| epochs:", EPOCHS, "| seed:", SEED)
if DEVICE == "cpu" and MODEL == "xlm-roberta-base":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")


wb = None
if USE_WANDB:
    try:
        try:
            import wandb
        except ImportError:
            import subprocess, sys
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
            import wandb
        wandb.login()     # Colab shows a box: paste the API key from wandb.ai/authorize
        wb = wandb.init(project="helaguide-c1", name=f"{RUN_TAG}-seed{SEED}",
                        config={"model": MODEL, "epochs": EPOCHS, "lr": LR, "batch": BATCH, "max_len": MAX_LEN,
                                "seed": SEED, "intent_window": INTENT_WINDOW, "run": RUN_TAG})
    except Exception as e:
        print("W&B not used:", e)
        wb = None


def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# Load data: upload the 3 files (or put them in /content/c1/ beforehand)
DATA_DIR = os.environ.get("C1_DATA", "/content/c1")
os.makedirs(DATA_DIR, exist_ok=True)
NEEDED = ["silver_tokens.jsonl", "gold_tokens.jsonl", "baseline_fasttext.json"]
OPTIONAL = ["fresh_tokens.jsonl"]          # your from-scratch test set, if ready
missing = [f for f in NEEDED if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    print("Please upload:", missing, "(and optionally", OPTIONAL, ")")
    from google.colab import files
    for name, content in files.upload().items():
        open(os.path.join(DATA_DIR, os.path.basename(name)), "wb").write(content)


def read_jsonl(name):
    with open(os.path.join(DATA_DIR, name), encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]


silver, gold = read_jsonl("silver_tokens.jsonl"), read_jsonl("gold_tokens.jsonl")
fresh = read_jsonl("fresh_tokens.jsonl") if os.path.exists(os.path.join(DATA_DIR, "fresh_tokens.jsonl")) else []
print("fresh test set:", f"{len(fresh)} turns" if fresh else "not uploaded (fresh scores will be skipped)")
baseline_path = os.path.join(DATA_DIR, "baseline_fasttext.json")
baseline = json.load(open(baseline_path, encoding="utf-8")) if os.path.exists(baseline_path) else {}

# dev split by DIALOGUE (never split one dialogue across train and dev)
set_seed()
dlg_ids = sorted({r["dialogue_id"] for r in silver})
random.shuffle(dlg_ids)
dev_ids = set(dlg_ids[: max(1, int(len(dlg_ids) * DEV_SHARE))])
train = [r for r in silver if r["dialogue_id"] not in dev_ids]
dev = [r for r in silver if r["dialogue_id"] in dev_ids]
print(f"silver turns: train {len(train)}, dev {len(dev)} | gold turns: {len(gold)}")
print("gold by code_mix:", collections.Counter(r["code_mix"] for r in gold))

In [ ]:
# Shared helpers: token-level datasets, training loop, metrics


def script_of(tokens):
    return "sinhala_script" if any("඀" <= c <= "෿" for t in tokens for c in t) else "romanized"


def group_of(r):
    return f"{r['code_mix']} / {script_of(r['tokens'])}"


tok = AutoTokenizer.from_pretrained(MODEL)


def encode_tokens(rows, field, label2id):
    out = []
    for r in rows:
        enc = tok(r["tokens"], is_split_into_words=True, truncation=True, max_length=MAX_LEN)
        labels, prev = [], None
        for w in enc.word_ids():
            if w is None or w == prev:
                labels.append(-100)          # special tokens and continuation pieces are ignored
            else:
                labels.append(label2id.get(r[field][w], -100))
            prev = w
        out.append({"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels})
    return out


def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    ids = [b["input_ids"] + [pad] * (n - len(b["input_ids"])) for b in batch]
    att = [b["attention_mask"] + [0] * (n - len(b["attention_mask"])) for b in batch]
    lab = [b["labels"] + [-100] * (n - len(b["labels"])) if isinstance(b["labels"], list) else b["labels"]
           for b in batch]
    return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(att), "labels": torch.tensor(lab)}


def train_model(model, train_data, dev_fn, epochs=EPOCHS, name="model"):
    """Plain PyTorch loop. Keeps the epoch with the best dev score (dev = synthetic, never gold)."""
    set_seed()
    model.to(DEVICE)
    dl = DataLoader(train_data, batch_size=BATCH, shuffle=True, collate_fn=collate)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = len(dl) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    best, best_state, history = -1, None, []
    for ep in range(1, epochs + 1):
        model.train(); t0, total = time.time(), 0.0
        for b in dl:
            b = {k: v.to(DEVICE) for k, v in b.items()}
            loss = model(**b).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            total += loss.item()
        score = dev_fn(model)
        history.append({"epoch": ep, "loss": round(total / len(dl), 4), "dev": round(score, 4)})
        print(f"  epoch {ep}: loss {total / len(dl):.4f}  dev {score:.4f}  ({time.time() - t0:.0f}s)")
        if wb:
            wb.log({f"{name}/train_loss": total / len(dl), f"{name}/dev": score, f"{name}/epoch": ep})
        if score > best:
            best = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, history


@torch.no_grad()
def predict_tokens(model, rows, id2label):
    """One label per word (from its first sub-token)."""
    model.eval(); preds = []
    for i in range(0, len(rows), 32):
        chunk = rows[i:i + 32]
        enc = tok([r["tokens"] for r in chunk], is_split_into_words=True, truncation=True,
                  max_length=MAX_LEN, padding=True, return_tensors="pt")
        logits = model(input_ids=enc["input_ids"].to(DEVICE), attention_mask=enc["attention_mask"].to(DEVICE)).logits
        best = logits.argmax(-1).cpu().tolist()
        for j, r in enumerate(chunk):
            words, prev = {}, None
            for k, w in enumerate(enc.word_ids(j)):
                if w is not None and w != prev:
                    words[w] = id2label[best[j][k]]
                prev = w
            preds.append([words.get(w, "O") for w in range(len(r["tokens"]))])   # truncated words -> "O"
    return preds


def spans(tags):
    """BIO tags -> set of (start, end, type) spans."""
    out, start, typ = set(), None, None
    for i, t in enumerate(list(tags) + ["O"]):
        if t.startswith("B-") or t == "O" or (t.startswith("I-") and t[2:] != typ):
            if typ is not None:
                out.add((start, i, typ))
            start, typ = (i, t[2:]) if t != "O" else (None, None)
    return out


def span_prf(golds, preds):
    per = collections.defaultdict(lambda: [0, 0, 0])   # tp, fp, fn
    for g, p in zip(golds, preds):
        gs, ps = spans(g), spans(p)
        for s in ps & gs: per[s[2]][0] += 1
        for s in ps - gs: per[s[2]][1] += 1
        for s in gs - ps: per[s[2]][2] += 1

    def prf(tp, fp, fn):
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        return pr, rc, (2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    tot = [sum(v[i] for v in per.values()) for i in range(3)]
    report = {k: dict(zip(["precision", "recall", "f1"], [round(x, 4) for x in prf(*v)]), support=v[0] + v[2])
              for k, v in sorted(per.items())}
    p, r, f = prf(*tot)
    report["MICRO"] = {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f, 4), "support": tot[0] + tot[2]}
    return report

def switch_flags(tags):
    """For each SI/EN word: True if the nearest SI/EN word before or after it has the
    other language (a code-switch point), False if inside a single-language run, None otherwise."""
    idx = [i for i, t in enumerate(tags) if t in ("SI", "EN")]
    flags = [None] * len(tags)
    for k, i in enumerate(idx):
        nb = [tags[idx[k - 1]]] if k > 0 else []
        nb += [tags[idx[k + 1]]] if k + 1 < len(idx) else []
        flags[i] = any(n != tags[i] for n in nb)
    return flags


def switch_accuracy(rows, preds):
    out = collections.defaultdict(list)
    for r, pr in zip(rows, preds):
        for g, p, f in zip(r["lang_tags"], pr, switch_flags(r["lang_tags"])):
            if f is not None:
                out["at_switch" if f else "inside_run"].append(g == p)
    return {k: {"acc": round(sum(v) / len(v), 4), "n": len(v)} for k, v in sorted(out.items())}


def label_prf(golds, preds, labels):
    """Per-label precision/recall/F1 over parallel flat lists."""
    rep_ = {}
    for l in labels:
        tp = sum(g == l and p == l for g, p in zip(golds, preds))
        fp = sum(g != l and p == l for g, p in zip(golds, preds))
        fn = sum(g == l and p != l for g, p in zip(golds, preds))
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        rep_[l] = {"precision": round(pr, 4), "recall": round(rc, 4),
                   "f1": round(2 * pr * rc / (pr + rc), 4) if pr + rc else 0.0, "support": tp + fn}
    rep_["MACRO_F1"] = round(sum(v["f1"] for v in rep_.values()) / len(labels), 4)
    return rep_


def print_prf(rep_, title):
    print(title)
    print(f"  {'label':24s}{'P':>8}{'R':>8}{'F1':>8}{'n':>6}")
    for k, v in rep_.items():
        if k != "MACRO_F1":
            print(f"  {k:24s}{v['precision']:>8}{v['recall']:>8}{v['f1']:>8}{v['support']:>6}")
    print(f"  macro-F1: {rep_['MACRO_F1']}")


def print_confusion(golds, preds, labels):
    m = {g: {p: 0 for p in labels} for g in labels}
    for g, p in zip(golds, preds):
        if g in m and p in m[g]:
            m[g][p] += 1
    short = {l: l[:9] for l in labels}
    print("  confusion (rows = true, columns = predicted)")
    print("  " + " " * 22 + "".join(f"{short[l]:>10}" for l in labels))
    for g in labels:
        print(f"  {g:22s}" + "".join(f"{m[g][p]:>10}" for p in labels))
    return m

## A. Language identification (token level)

In [ ]:
LANG = ["SI", "EN", "MIXED", "NE", "OTHER"]
l2i = {l: i for i, l in enumerate(LANG)}; i2l = {i: l for l, i in l2i.items()}
gold_lang = [r for r in gold if "UNLABELLED" not in r["lang_tags"]]


def lang_dev(model):
    pred = predict_tokens(model, dev, i2l)
    pairs = [(g, p) for r, pr in zip(dev, pred) for g, p in zip(r["lang_tags"], pr)]
    return sum(g == p for g, p in pairs) / max(1, len(pairs))


set_seed()
lang_model = AutoModelForTokenClassification.from_pretrained(
    MODEL, num_labels=len(LANG), id2label=i2l, label2id=l2i)
print("Training language ID...")
lang_model, lang_hist = train_model(lang_model, encode_tokens(train, "lang_tags", l2i), lang_dev, name="lang")

lang_pred = predict_tokens(lang_model, gold_lang, i2l)
acc_all, acc_sien, by_group = [], [], collections.defaultdict(list)
confusion = collections.Counter()
for r, pr in zip(gold_lang, lang_pred):
    for g, p in zip(r["lang_tags"], pr):
        acc_all.append(g == p)
        confusion[(g, p)] += 1
        if g in ("SI", "EN"):                   # same subset as the fastText baseline
            acc_sien.append(g == p)
            by_group[group_of(r)].append(g == p)
            by_group[f"gold={g}"].append(g == p)
mean = lambda x: round(sum(x) / len(x), 4) if x else None
lang_results = {
    "all_tags_accuracy": mean(acc_all), "si_en_accuracy": mean(acc_sien),
    "si_en_by_group": {k: {"acc": mean(v), "n": len(v)} for k, v in sorted(by_group.items())},
    "confusion_gold_to_pred": {f"{g}->{p}": n for (g, p), n in confusion.most_common() if g != p},
    "history": lang_hist,
}

ft = baseline.get("fasttext_token_accuracy", {})
print(f"\nGold set, SI/EN tokens  (n={len(acc_sien)})")
print(f"{'group':32s}{'fastText':>10}{'XLM-R':>10}")
for k in ["ALL"] + sorted(by_group):
    ours = mean(acc_sien) if k == "ALL" else mean(by_group[k])
    base = ft.get(k, {}).get("acc")
    print(f"{k:32s}{(base if base is not None else '-'):>10}{ours:>10}")
print(f"\nAll 5 tags accuracy: {lang_results['all_tags_accuracy']}")
flat_g = [g for r in gold_lang for g in r["lang_tags"]]
flat_p = [p for pr in lang_pred for p in pr]
lang_results["per_tag"] = label_prf(flat_g, flat_p, LANG)
print_prf(lang_results["per_tag"], "\nPer-tag scores (gold):")
lang_results["switch_points"] = switch_accuracy(gold_lang, lang_pred)
print("Accuracy at code-switch points vs inside single-language runs (gold, SI/EN words):",
      lang_results["switch_points"])
print("Most common errors:", dict(list(lang_results["confusion_gold_to_pred"].items())[:8]))

## B. Entities (BIO, span-level F1)

In [ ]:
ENT_TYPES = sorted({t[2:] for r in silver + gold for t in r["ner_tags"] if t != "O"})
NER = ["O"] + [f"{p}-{t}" for t in ENT_TYPES for p in ("B", "I")]
n2i = {l: i for i, l in enumerate(NER)}; i2n = {i: l for l, i in n2i.items()}
train_types = {t[2:] for r in train for t in r["ner_tags"] if t != "O"}
print("entity types in training data:", sorted(train_types))
print("entity types only in gold (cannot be learned yet):", sorted(set(ENT_TYPES) - train_types))


def ner_dev(model):
    return span_prf([r["ner_tags"] for r in dev], predict_tokens(model, dev, i2n))["MICRO"]["f1"]


set_seed()
ner_model = AutoModelForTokenClassification.from_pretrained(
    MODEL, num_labels=len(NER), id2label=i2n, label2id=n2i)
print("Training entities...")
ner_model, ner_hist = train_model(ner_model, encode_tokens(train, "ner_tags", n2i), ner_dev, name="ner")

ner_pred = predict_tokens(ner_model, gold, i2n)
ner_all = span_prf([r["ner_tags"] for r in gold], ner_pred)
learnable = [[t if t == "O" or t[2:] in train_types else "O" for t in r["ner_tags"]] for r in gold]
ner_learnable = span_prf(learnable, ner_pred)
ner_results = {"all_types": ner_all, "learnable_types_only": ner_learnable, "history": ner_hist}
print("\nGold set entity scores (span-level):")
print(f"{'type':20s}{'P':>8}{'R':>8}{'F1':>8}{'n':>6}")
for k, v in ner_all.items():
    print(f"{k:20s}{v['precision']:>8}{v['recall']:>8}{v['f1']:>8}{v['support']:>6}")
print(f"\nMicro F1 on types present in training data: {ner_learnable['MICRO']['f1']}")

## C. Intent (last 3 user messages, window_3)
Intent is scored on the gold turns whose intent appears in the training data. With the support-intent
dialogues added, that is all 12 intents (all 142 gold turns); any intent still missing is reported as a gap.

In [ ]:
def with_history(rows):
    by_dlg = collections.defaultdict(list)
    for r in rows:
        by_dlg[r["dialogue_id"]].append(r)
    out = []
    for d in by_dlg.values():
        d.sort(key=lambda r: r["turn"])
        for i, r in enumerate(d):
            ctx = d[: i + 1] if INTENT_WINDOW <= 0 else d[max(0, i + 1 - INTENT_WINDOW): i + 1]
            out.append({**r, "history": " </s> ".join(x["text"] for x in ctx)})
    return out


tok_int = AutoTokenizer.from_pretrained(MODEL)
tok_int.truncation_side = "left"     # if a long input must be cut, drop the OLDEST text
INTENTS = sorted({r["intent"] for r in train})
t2i = {l: i for i, l in enumerate(INTENTS)}; i2t = {i: l for l, i in t2i.items()}
gold_h = with_history(gold)
gold_core = [r for r in gold_h if r["intent"] in t2i]
not_covered = collections.Counter(r["intent"] for r in gold_h if r["intent"] not in t2i)
print(f"intents learned: {INTENTS}")
print(f"gold turns scored: {len(gold_core)}; not covered by training data: {dict(not_covered)}")


def encode_seq(rows):
    out = []
    for r in rows:
        enc = tok_int(r["history"], truncation=True, max_length=MAX_LEN)   # cut from the left: keep the newest text
        out.append({"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": t2i[r["intent"]]})
    return out


@torch.no_grad()
def predict_seq(model, rows):
    model.eval(); preds = []
    for i in range(0, len(rows), 32):
        enc = tok_int([r["history"] for r in rows[i:i + 32]], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors="pt")
        logits = model(input_ids=enc["input_ids"].to(DEVICE), attention_mask=enc["attention_mask"].to(DEVICE)).logits
        preds += [i2t[x] for x in logits.argmax(-1).cpu().tolist()]
    return preds


def macro_f1(golds, preds, labels):
    f1s = []
    for l in labels:
        tp = sum(g == l and p == l for g, p in zip(golds, preds))
        fp = sum(g != l and p == l for g, p in zip(golds, preds))
        fn = sum(g == l and p != l for g, p in zip(golds, preds))
        pr = tp / (tp + fp) if tp + fp else 0.0
        rc = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(f1s) / len(f1s)


dev_h = with_history(dev)


def intent_dev(model):
    return macro_f1([r["intent"] for r in dev_h], predict_seq(model, dev_h), INTENTS)


set_seed()
int_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(INTENTS), id2label=i2t, label2id=t2i)
print("Training intent...")
int_model, int_hist = train_model(int_model, encode_seq(with_history(train)), intent_dev, name="intent")

int_pred = predict_seq(int_model, gold_core)
g_int = [r["intent"] for r in gold_core]
int_by_group = collections.defaultdict(list)
for r, p in zip(gold_core, int_pred):
    int_by_group[group_of(r)].append(r["intent"] == p)
int_results = {
    "accuracy": mean([g == p for g, p in zip(g_int, int_pred)]),
    "macro_f1": round(macro_f1(g_int, int_pred, INTENTS), 4),
    "by_group": {k: {"acc": mean(v), "n": len(v)} for k, v in sorted(int_by_group.items())},
    "errors": dict(collections.Counter(f"{g}->{p}" for g, p in zip(g_int, int_pred) if g != p).most_common(10)),
    "gold_turns_scored": len(gold_core), "gold_intents_not_covered": dict(not_covered), "history": int_hist,
}
print(f"\nIntent accuracy {int_results['accuracy']}  macro-F1 {int_results['macro_f1']}  (n={len(gold_core)})")
for k, v in int_results["by_group"].items():
    print(f"  {k:32s}{v['acc']:>8}  n={v['n']}")
print("Most common confusions:", int_results["errors"])
int_results["per_intent"] = label_prf(g_int, int_pred, INTENTS)
print_prf(int_results["per_intent"], "\nPer-intent scores (gold, XLM-R):")
int_results["confusion"] = print_confusion(g_int, int_pred, INTENTS)

# Non-neural baseline: TF-IDF (character + word n-grams) + logistic regression, same input (window_3)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, make_union
tfidf = make_pipeline(
    make_union(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True),
               TfidfVectorizer(analyzer="word", ngram_range=(1, 2), sublinear_tf=True)),
    LogisticRegression(max_iter=3000, C=10))
train_h = with_history(train)
tfidf.fit([r["history"] for r in train_h], [r["intent"] for r in train_h])
tf_pred = list(tfidf.predict([r["history"] for r in gold_core]))
int_results["tfidf_baseline"] = {
    "accuracy": mean([g == p for g, p in zip(g_int, tf_pred)]),
    "macro_f1": round(macro_f1(g_int, tf_pred, INTENTS), 4)}
print(f"\nTF-IDF + logistic regression baseline (gold): accuracy {int_results['tfidf_baseline']['accuracy']}"
      f"  macro-F1 {int_results['tfidf_baseline']['macro_f1']}   vs XLM-R {int_results['accuracy']} / {int_results['macro_f1']}")

## D. Fresh test set (written and labelled from scratch, no pre-labels)
The fairest comparison: neither the heuristic nor an LLM influenced these labels.

In [ ]:
fresh_results = {}
if fresh:
    fr = [r for r in fresh if "UNLABELLED" not in r["lang_tags"]]
    pred = predict_tokens(lang_model, fr, i2l)
    systems = {"heuristic": [r.get("heuristic_tags") for r in fr],
               "fastText": [r.get("fasttext_tags") for r in fr], "XLM-R": pred}
    table = collections.defaultdict(lambda: collections.defaultdict(list))
    for name, preds in systems.items():
        if any(p is None for p in preds):
            continue
        for r, pr in zip(fr, preds):
            for g, p in zip(r["lang_tags"], pr):
                if g in ("SI", "EN"):
                    for k in ("ALL", group_of(r), f"gold={g}"):
                        table[name][k].append(g == p)
    names = list(table)
    keys = sorted({k for n in names for k in table[n]}, key=lambda k: (k != "ALL", k))
    print(f"Fresh set, SI/EN words (n={len(table['XLM-R']['ALL'])})")
    print(f"{'group':32s}" + "".join(f"{n:>11}" for n in names))
    for k in keys:
        print(f"{k:32s}" + "".join(f"{(mean(table[n][k]) if table[n][k] else '-'):>11}" for n in names))
    lang_all = mean([g == p for r, pr in zip(fr, pred) for g, p in zip(r["lang_tags"], pr) if g != "UNLABELLED"])
    xl_err = collections.Counter(f"{g}->{p}" for r, pr in zip(fr, pred) for g, p in zip(r["lang_tags"], pr) if g != p)
    wrong_words = [(w, g, p) for r, pr in zip(fr, pred) for w, g, p in zip(r["tokens"], r["lang_tags"], pr) if g != p]
    print(f"XLM-R all-tags accuracy: {lang_all}")
    fresh_switch = {n: switch_accuracy(fr, systems[n]) for n in names}
    print("Accuracy at code-switch points vs inside runs (fresh, SI/EN words):")
    for n, v in fresh_switch.items():
        print(f"  {n:10s}", v)
    fresh_per_tag = label_prf([g for r in fr for g in r["lang_tags"]], [p for pr in pred for p in pr], LANG)
    print_prf(fresh_per_tag, "Per-tag scores (fresh, XLM-R):")
    print("XLM-R wrong words (word, gold, predicted):", wrong_words[:25])

    f_ner = span_prf([r["ner_tags"] for r in fresh], predict_tokens(ner_model, fresh, i2n))
    print(f"\nFresh entities micro-F1: {f_ner['MICRO']['f1']}  (n={f_ner['MICRO']['support']})")
    for k, v in f_ner.items():
        if k != "MICRO":
            print(f"  {k:20s}P {v['precision']}  R {v['recall']}  F1 {v['f1']}  n={v['support']}")

    fh = [r for r in with_history(fresh) if r["intent"] in t2i]
    fp = predict_seq(int_model, fh)
    f_int_acc = mean([r["intent"] == p for r, p in zip(fh, fp)])
    print(f"\nFresh intent accuracy: {f_int_acc} (n={len(fh)})")
    print("Errors:", [(r["dialogue_id"], r["turn"], r["intent"], p) for r, p in zip(fh, fp) if r["intent"] != p])
    f_g = [r["intent"] for r in fh]
    fresh_int_prf = label_prf(f_g, fp, INTENTS)
    print_prf(fresh_int_prf, "Per-intent scores (fresh, XLM-R):")
    fresh_conf = print_confusion(f_g, fp, INTENTS)
    f_tf = list(tfidf.predict([r["history"] for r in fh]))
    fresh_tfidf = {"accuracy": mean([g == p for g, p in zip(f_g, f_tf)]), "macro_f1": round(macro_f1(f_g, f_tf, INTENTS), 4)}
    print(f"TF-IDF baseline (fresh): accuracy {fresh_tfidf['accuracy']}  macro-F1 {fresh_tfidf['macro_f1']}"
          f"   vs XLM-R {f_int_acc} / {round(macro_f1(f_g, fp, INTENTS), 4)}")
    fresh_results = {
        "language_id_si_en": {n: {k: {"acc": mean(v), "n": len(v)} for k, v in table[n].items()} for n in names},
        "xlmr_all_tags_accuracy": lang_all, "xlmr_errors": dict(xl_err.most_common(15)),
        "entities": f_ner, "intent_accuracy": f_int_acc, "intent_n": len(fh),
        "lang_switch_points": fresh_switch, "lang_per_tag": fresh_per_tag,
        "intent_per_intent": fresh_int_prf, "intent_confusion": fresh_conf, "intent_tfidf_baseline": fresh_tfidf,
    }
else:
    print("No fresh_tokens.jsonl uploaded; skipping.")

## E. Save the trained models (for the API and the context experiments)
Saved in half precision (~0.55 GB each) to Google Drive: MyDrive/c1_models/run_v2_seed<SEED>/{lid,ner,intent}.
Colab asks for permission to access Drive the first time.

In [ ]:
saved_to = None
if SAVE_MODELS:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        base = "/content/drive/MyDrive/c1_models"
    except Exception as e:
        base = os.path.join(DATA_DIR, "c1_models")      # not on Colab: save next to the data
        print("WARNING: Google Drive not mounted, so the models are saved to temporary storage and will be lost "
              "when Colab closes:", e)
    saved_to = os.path.join(base, f"run_{RUN_TAG}_seed{SEED}")
    for name, m in [("lid", lang_model), ("ner", ner_model), ("intent", int_model)]:
        d = os.path.join(saved_to, name)
        m.half().save_pretrained(d)
        tok.save_pretrained(d)
        m.float()
    json.dump({"base_model": MODEL, "seed": SEED, "lang_labels": LANG, "ner_labels": NER, "intents": INTENTS,
               "max_len": MAX_LEN, "history_separator": " </s> ", "intent_window": INTENT_WINDOW,
               "truncation_side": "left", "run": RUN_TAG,
               "input": f"intent model reads the last {INTENT_WINDOW} user messages" if INTENT_WINDOW > 0
               else "intent model reads all user turns so far"},
              open(os.path.join(saved_to, "c1_config.json"), "w", encoding="utf-8"), indent=2)
    print("models saved to", saved_to)
else:
    print("SAVE_MODELS is off; models not saved.")

## Save results
Downloads `xlmr_results_v2_seed<SEED>.json`. Put it in `research/results/` and commit it.
The trained models are not saved (they are large); rerun this notebook to reproduce.

In [ ]:
results = {
    "run_at": time.strftime("%Y-%m-%d %H:%M"), "model": MODEL, "models_saved_to": saved_to,
    "settings": {"epochs": EPOCHS, "batch": BATCH, "lr": LR, "max_len": MAX_LEN, "seed": SEED, "dev_share": DEV_SHARE,
                 "intent_window": INTENT_WINDOW, "truncation_side": "left", "run": RUN_TAG},
    "data": {"silver_train_turns": len(train), "silver_dev_turns": len(dev), "gold_turns": len(gold), "fresh_turns": len(fresh)},
    "language_id": lang_results, "entities": ner_results, "intent": int_results,
    "fresh_test": fresh_results,
}
out = os.path.join(DATA_DIR, f"xlmr_results_{RUN_TAG}_seed{SEED}.json")
json.dump(results, open(out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("saved", out)
if wb:
    wb.summary.update({"lang/fresh_si_en": (fresh_results.get("language_id_si_en", {}).get("XLM-R", {}).get("ALL", {}).get("acc")),
                       "lang/gold_si_en": lang_results["si_en_accuracy"], "intent/gold_macro_f1": int_results["macro_f1"],
                       "intent/gold_tfidf_macro_f1": int_results["tfidf_baseline"]["macro_f1"],
                       "ner/gold_micro_f1": ner_results["all_types"]["MICRO"]["f1"],
                       "intent/fresh_acc": fresh_results.get("intent_accuracy")})
    wb.save(out)
    wb.finish()
try:
    from google.colab import files
    files.download(out)
except ImportError:
    pass